# Fixed penalty caps from a revert-rate target

**Inputs** (parameters in the next cell): exclusivity window `T_EXCL`, target revert rate.
**Output**: two protocol-wide fixed penalty caps in bps of order size — one for **correlated**
pairs, one for **uncorrelated** pairs — **fitted on June 2026 prices** and **evaluated
out-of-sample on July 2026**, over the top `TOP_PAIRS` pairs per group of CoW flow pooled
across all supported chains.

The model, in three sentences:

1. A solver that won an auction holds exclusivity for `T_EXCL` seconds. If the price moves
   against it by more than the cap during that window, reverting is cheaper than settling — so,
   for one pair, the expected *price-driven* revert rate at a fixed cap `c` is the share of
   historical `T_EXCL`-second price moves worse than `-c`.
2. CoW settlement attempts, pooled across chains, give the weights: the expected revert rate of
   a fixed cap is the attempt-weighted average of the per-pair rates.
3. Pairs are **correlated** when both legs sit in the same CoW correlated-token bucket (the CMS
   lists behind the reduced fee), **uncorrelated** otherwise. Each group's cap is the smallest
   `c` whose expected group revert rate on the **fit period (June)** is at or below the target.

Pair identity is deliberately coarse:

- tokens in the same CMS bucket are treated as one — they all resolve to the same price book,
  so e.g. `wstETH→COW` counts toward the same pair as `WETH→COW` (both are `ETH↔COW`);
- **both directions of a pair are always combined**: one `ETH↔USDC` pair carries the attempts
  of `WETH→USDC` and `USDC→WETH` alike, its revert curve the attempt-weighted mix of the two
  directions' adverse tails.

**Data provenance** — two sources only, both fetched and cached under `../data/` (delete a
cached file to refresh it):

- **CoW analytics DB** (permissioned): the `data/{chain}_*.csv` extracts, created via
  `scripts/fetch_orderbook_data.py` when missing — the DB-only fetcher, which never touches
  Dune. Pre-existing full extracts from `fetch_penalties_data.py` are read as-is; the four
  Dune-sourced columns they carry (`order_size_usd`, `markout_*`, `execution_cost_native`)
  are **not** used. Every column used here (`sell_token`, `buy_token`, the three filter
  flags, `volume_native`, `penalty_cap_native`, `auction_timestamp`) is DB-sourced; USD
  sizes below are derived as `volume_native x` Binance daily native-token close. The CMS
  correlated-token lists are CoW infrastructure as well.
- **Binance** (permissionless): `exchangeInfo` for the listing universe, 1-second klines for
  the price history, daily klines for the native-token USD closes.

No Dune, no third-party token lists.

Deliberate simplifications, each discussed in the caveats at the end:

1. price-driven reverts only — technical/external reverts sit on top of this target;
2. no bidding response — a revert happens iff the adverse move exceeds the cap;
3. Binance prices (1-second klines) proxy on-chain executable prices;
4. direct percentile method — the cap is a quantile of historical moves, no volatility model.

In [ ]:
import glob
import json
import os
import re
import subprocess
import sys
import urllib.error
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

T_EXCL = 26                 # exclusivity window in seconds
TARGET_REVERT_RATE = 0.08   # tolerated price-driven revert rate per settlement attempt
TOP_PAIRS = 50              # pairs per group entering the calibration, by attempts

COW_START, COW_END = "2026-01-01", "2026-06-22"   # attempt window (only used when fetching)
FIT_DAYS = [d.strftime("%Y-%m-%d") for d in pd.date_range("2026-06-01", "2026-06-30")]
TEST_DAYS = [d.strftime("%Y-%m-%d") for d in pd.date_range("2026-07-01", "2026-07-30")]

# chain config (mirrors scripts/fetch_penalties_data.py and the CMS bucket naming)
NATIVE = {"ethereum": "ETH", "arbitrum": "ETH", "base": "ETH", "gnosis": "XDAI",
          "polygon": "POL", "bnb": "BNB", "avalanche_c": "AVAX"}
CMS_NET = {"ethereum": "MAINNET", "arbitrum": "ARBITRUM", "base": "BASE", "gnosis": "GNOSIS",
           "polygon": "POLYGON", "bnb": "BNB", "avalanche_c": "AVALANCHE"}

# Tokens outside the CMS buckets that carry real flow, mapped to the Binance book that
# proxies their price. Informed by the concentration analysis in section 2 — extend this
# dict when the "unresolved tokens" list printed there grows a new head. Addresses are
# unique across chains in practice (cbBTC deliberately shares one address on ethereum/base).
HARDCODED_BOOK = {
    # BTC wrappers -> the deep BTC book
    "0x2260fac5e5542a773aa44fbcfedf7c193bc2c599": "BTC",   # WBTC (ethereum)
    "0x2f2a2543b76a4166549f7aab2e75bef0aefc5b0f": "BTC",   # WBTC (arbitrum)
    "0x1bfd67037b42cf73acf2047067bd4f2c47d9bfd6": "BTC",   # WBTC (polygon)
    "0xcbb7c0000ab88b473b1f5afd9ef808440eed33bf": "BTC",   # cbBTC (ethereum & base)
    "0x18084fba666a33d37592fa2633fd49a74dd93a88": "BTC",   # tBTC (ethereum)
    "0x8236a87084f8b84306f72007f36f2618a5634494": "BTC",   # LBTC (ethereum)
    "0x5ee5bf7ae06d1be5997a1a72006fe6c607ec6de8": "BTC",   # aEthWBTC (ethereum)
    "0x7130d2a12b9bcbfae4f2634d864a1ee1ce3ead9c": "BTC",   # BTCB (bnb)
    # tokens with their own Binance listing
    "0x514910771af9ca656af840dff83e8264ecf986ca": "LINK",  # (ethereum)
    "0x7fc66500c84a76ad7e9c93437bfc5ac33e2ddae9": "AAVE",  # (ethereum)
    "0x1f9840a85d5af5bf1d1762f925bdaddc4201f984": "UNI",   # (ethereum)
    "0x45804880de22913dafe09f4980848ece6ecbaf78": "PAXG",  # (ethereum)
    "0x68749665ff8d2d112fa859aa293f07a622782f38": "XAUT",  # (ethereum)
    "0x77e06c9eccf2e797fd462a92b6d7642ef85b0a44": "TAO",   # wTAO (ethereum)
    "0x56072c95faa701256059aa122697b133aded9279": "SKY",   # (ethereum)
    "0xd533a949740bb3306d119cc777fa900ba034cd52": "CRV",   # (ethereum)
    "0x6982508145454ce325ddbe47a25d4ec3d2311933": "PEPE",  # (ethereum)
    "0xfaba6f8e4a5e8ab82f62fe7c39859fa577269be3": "ONDO",  # (ethereum)
    "0xdef1ca1fb7fbcdc777520aa7f396b4e015f497ab": "COW",   # (ethereum)
    "0xbb4cdb9cbd36b01bd1cbaebf2de08d9173bc095c": "BNB",   # WBNB (bnb)
    "0x9c58bacc331c9aa871afd802db6379a98e80cedb": "GNO",   # (gnosis)
    "0x940181a94a35a4569e4529a3cdfb74e38fd98631": "AERO",  # (base)
    # yield-bearing USD, drifts vs USD at the savings rate only
    "0xa3931d71877c0e7a3148cb7eb4463524fec27fbd": "USDC",  # sUSDS (ethereum)
}

KLINE_CACHE = "../data/binance_klines_1s"
os.makedirs(KLINE_CACHE, exist_ok=True)


def cached_json(path, url):
    if not os.path.exists(path):        # delete the cached file to refresh it
        req = urllib.request.Request(url, headers={"User-Agent": "penalty-research-notebook"})
        with urllib.request.urlopen(req) as r, open(path, "wb") as f:
            f.write(r.read())
    with open(path) as f:
        return json.load(f)

## 1. Token pairs traded on CoW, all chains

One row of a chain CSV is one settlement attempt (an order inside an auction's winning
solution). We keep the standard frame used across this repo — fill-or-kill, in-market,
penalty-eligible — for every supported chain and pool the attempts.

Attempts, not USD volume, are the weight: the revert rate we target is a rate *per attempt*,
and attempts are observed for settled and reverted rows alike. The weights come from the full
extract window (the extracts end before July); the price data is what splits into a June fit
and a July test. A USD size per attempt is still computed — `volume_native x` the day's
Binance native-token close (xDAI ~ 1 USD) — for the concentration analysis and the
current-cap comparison; a handful of rows with corrupt `volume_native` (implying > $1B
orders) get their USD size dropped.

In [ ]:
def window_days(path):
    """Days covered by an extract, read off its {chain}_{start}_{end}[_db].csv name."""
    start, end = re.findall(r"\d{4}-\d{2}-\d{2}", os.path.basename(path))[:2]
    return (pd.Timestamp(end) - pd.Timestamp(start)).days


frames = []
for chain in NATIVE:
    pattern = f"../data/{chain}_????-??-??_????-??-??*.csv"
    files = glob.glob(pattern)
    if not files:   # DB-only fetch, no Dune (needs ANALYTICS_DB_URL in ../.env)
        subprocess.run([sys.executable, "../scripts/fetch_orderbook_data.py", "--chain", chain,
                        "--start", COW_START, "--end", COW_END], check=True)
        files = glob.glob(pattern)
    df = pd.read_csv(max(files, key=window_days), low_memory=False,
                     usecols=["sell_token", "buy_token", "partially_fillable",
                              "is_out_of_market", "is_excluded_from_penalties",
                              "volume_native", "penalty_cap_native", "auction_timestamp"])
    df = df[df.partially_fillable.eq(False)
            & df.is_out_of_market.eq(False)   # rows with NaN flags drop out
            & df.is_excluded_from_penalties.eq(False)].copy()
    df["chain"] = chain
    frames.append(df)
att = pd.concat(frames, ignore_index=True)
att["sell_token"] = att.sell_token.str.lower()
att["buy_token"] = att.buy_token.str.lower()
att["day"] = pd.to_datetime(att.auction_timestamp, utc=True, format="mixed").dt.strftime("%Y-%m-%d")

start_ms = int(pd.Timestamp(COW_START, tz="UTC").timestamp() * 1000)
PX1D = {}
for asset in set(NATIVE.values()) - {"XDAI"}:
    kl = cached_json(f"../data/binance_1d_{asset}USDT.json",
                     f"https://api.binance.com/api/v3/klines?symbol={asset}USDT"
                     f"&interval=1d&startTime={start_ms}&limit=500")
    PX1D[asset] = {pd.Timestamp(k[0], unit="ms").strftime("%Y-%m-%d"): float(k[4]) for k in kl}
att["usd"] = [v / 1e18 * (1.0 if NATIVE[c] == "XDAI" else PX1D[NATIVE[c]].get(d, np.nan))
              for v, c, d in zip(att.volume_native, att.chain, att.day)]
att.loc[att.usd > 1e9, "usd"] = np.nan      # corrupt volume_native rows

pairs = (att.groupby(["chain", "sell_token", "buy_token"])
            .agg(attempts=("usd", "size"), usd=("usd", "sum")).reset_index())
print(att.groupby("chain").size().rename("attempts").to_string())
print(f"total: {len(att):,} attempts, {len(pairs):,} directed (chain, pair) rows, "
      f"${att.usd.sum() / 1e9:.1f}B")

## 2. Classify pairs, map them to price feeds, and pick the top pairs

**Classification** uses CoW's own correlated-token buckets from the CMS (one "Stables" and one
"WETH assets" bucket per chain): a pair is **correlated** iff both legs are in the same bucket
of its chain.

**Price feed**: every token is resolved to the Binance USDT book that proxies its price, and a
directed pair's price is the ratio `sell_book / buy_book` (USDT itself is the quote, i.e. a
constant 1). Resolution order per token:

1. the `HARDCODED_BOOK` map (tokens outside the CMS buckets that carry real flow);
2. its CMS symbol is listed on Binance, directly or minus a leading `W`
   (`WETH -> ETH`, `WPOL -> POL`);
3. it is in a CMS bucket → the bucket's **anchor**: the most CoW-traded bucket member that
   itself resolves (so `wstETH -> ETH`, `DAI -> USDC`, gnosis `xDAI -> USDC`, ...).

Step 3 is what makes bucket members interchangeable in pair identity: `wstETH→COW` and
`WETH→COW` both become the `ETH↔COW` pair. Because a directed pair's price is always
`sell/buy`, the adverse direction for the solver is always the **left tail**; the two
directions of a pair are then **combined** into one undirected pair whose curve is the
attempt-weighted mix of the two directional tails. Pairs whose legs resolve to the same book
(`WETH↔wstETH`, `DAI↔USDC`, `WBTC↔cbBTC`, ...) have no independent price and are excluded
from the curves but reported.

The cell also answers **how concentrated the flow is** (how many pooled pairs capture
80/90/95% of trades and of USD volume, and which unresolved tokens are next in line for
`HARDCODED_BOOK`), then selects the **top `TOP_PAIRS` pairs per group by attempts** as the
calibration set and prints what share of flow that covers.

In [ ]:
cms = cached_json("../data/cow_correlated_tokens.json",
                  "https://cms.cow.finance/api/correlated-tokens?pagination%5BpageSize%5D=100")
info = cached_json("../data/binance_exchangeinfo.json",
                   "https://api.binance.com/api/v3/exchangeInfo")
BASES = {s["baseAsset"] for s in info["symbols"]
         if s["quoteAsset"] == "USDT" and s["status"] == "TRADING"
         and s["isSpotTradingAllowed"]}


def binance_asset(sym):
    u = sym.upper()
    if u == "USDT" or u in BASES:       # USDT itself is the quote: constant price 1
        return u
    if len(u) > 2 and u.startswith("W") and u[1:] in BASES:
        return u[1:]                    # wrapped majors: WETH -> ETH, WPOL -> POL
    return None


BUCKETS = {chain: {b["attributes"]["name"]: {a.lower(): s
                                             for a, s in b["attributes"]["tokens"].items()}
                   for b in cms["data"] if f"for {CMS_NET[chain]}" in b["attributes"]["name"]}
           for chain in NATIVE}
SYMBOL = {}
for chain in NATIVE:
    SYMBOL[chain] = {}
    for tokens in BUCKETS[chain].values():
        SYMBOL[chain].update(tokens)
    SYMBOL[chain].setdefault("0x" + "e" * 40, NATIVE[chain])   # native-token sentinel

# bucket anchor = the most CoW-traded member that resolves to a Binance book
ANCHOR = {}
for chain in NATIVE:
    sub = att[att.chain == chain]
    traded = (pd.concat([sub.groupby("sell_token").size(), sub.groupby("buy_token").size()])
                .groupby(level=0).sum())
    for name, tokens in BUCKETS[chain].items():
        ANCHOR[(chain, name)] = next(
            (binance_asset(tokens[a]) for a in sorted(tokens, key=lambda a: -traded.get(a, 0))
             if binance_asset(tokens[a])), None)


def bucket_of(chain, addr):
    return next((n for n, tokens in BUCKETS[chain].items() if addr in tokens), None)


def resolve(chain, addr):
    """Binance asset whose USDT book proxies this token's price (None if none found)."""
    if addr in HARDCODED_BOOK:
        return HARDCODED_BOOK[addr]
    sym = SYMBOL[chain].get(addr)
    asset = binance_asset(sym) if sym else None
    if asset is None and bucket_of(chain, addr):
        asset = ANCHOR[(chain, bucket_of(chain, addr))]
    return asset


asset_s = [resolve(c, a) for c, a in zip(pairs.chain, pairs.sell_token)]
asset_b = [resolve(c, a) for c, a in zip(pairs.chain, pairs.buy_token)]
pairs["asset_s"], pairs["asset_b"] = asset_s, asset_b
pairs["group"] = ["correlated" if bucket_of(c, s) is not None
                  and bucket_of(c, s) == bucket_of(c, b) else "uncorrelated"
                  for c, s, b in zip(pairs.chain, pairs.sell_token, pairs.buy_token)]
pairs["feed"] = [f"{a}/{b}" if a is not None and b is not None and a != b else None
                 for a, b in zip(asset_s, asset_b)]

# --- concentration: how many pairs capture the flow? ---
key = ["|".join(sorted([a or f"{c[:3]}:{s[:8]}", b or f"{c[:3]}:{t[:8]}"]))
       for c, s, t, a, b in zip(pairs.chain, pairs.sell_token, pairs.buy_token,
                                asset_s, asset_b)]
conc = pairs.assign(key=key).groupby("key").agg(attempts=("attempts", "sum"),
                                                usd=("usd", "sum"))
for metric in ("attempts", "usd"):
    s = conc[metric].sort_values(ascending=False)
    cum = s.cumsum() / s.sum()
    n80, n90, n95 = (int((cum < q).sum()) + 1 for q in (0.80, 0.90, 0.95))
    print(f"{metric:8s}: 80% of flow -> {n80:4d} pairs | 90% -> {n90:4d} | 95% -> {n95:4d}"
          f"   (universe: {len(s):,} pooled undirected pairs)")

total_att, total_usd = pairs.attempts.sum(), pairs.usd.sum()
mapped = pairs[pairs.feed.notna()].copy()
same_book = pairs.asset_s.notna() & (pairs.asset_s == pairs.asset_b)
print(f"\nmapped to a usable feed: {mapped.attempts.sum() / total_att:.1%} of attempts, "
      f"{mapped.usd.sum() / total_usd:.1%} of USD volume")
print(f"same-book pairs excluded: {pairs[same_book].attempts.sum() / total_att:.1%} of attempts, "
      f"{pairs[same_book].usd.sum() / total_usd:.1%} of USD volume")

unres = pd.concat([
    pairs[pairs.asset_s.isna()].rename(columns={"sell_token": "token"}),
    pairs[pairs.asset_b.isna()].rename(columns={"buy_token": "token"})])
top_unres = (unres.groupby(["chain", "token"]).agg(attempts=("attempts", "sum"),
                                                   usd=("usd", "sum"))
                  .sort_values("usd", ascending=False).head(10))
print("\nnext candidates for HARDCODED_BOOK (top unresolved tokens by USD):")
print(top_unres.round(0).to_string())

# --- undirected pairs and top-N selection ---
mapped["ukey"] = ["|".join(sorted([a, b])) for a, b in zip(mapped.asset_s, mapped.asset_b)]
mapped["direction"] = np.where(mapped.asset_s < mapped.asset_b, "fwd", "rev")
upairs = (mapped.pivot_table(index=["ukey", "group"], columns="direction",
                             values="attempts", aggfunc="sum", fill_value=0)
                .reindex(columns=["fwd", "rev"], fill_value=0).reset_index())
upairs.columns.name = None
upairs["attempts"] = upairs.fwd + upairs.rev
usable = (upairs.sort_values("attempts", ascending=False)
                .groupby("group", sort=False).head(TOP_PAIRS).reset_index(drop=True))
usable["pair"] = usable.ukey.str.replace("|", "↔", regex=False)

print(f"\nselected top {TOP_PAIRS} pairs per group "
      f"({usable.groupby('group').size().to_dict()} available):")
print(usable.groupby("group").attempts.sum().to_string())
print(f"selection covers {usable.attempts.sum() / total_att:.1%} of all attempts "
      f"({usable.attempts.sum() / mapped.attempts.sum():.1%} of mapped attempts)")

## 3. Price moves over the exclusivity window, June and July

For each Binance book: 1-second klines (close per second), forward-filled onto the full per-day
1-second grid. A directed pair's per-day price grid is the ratio of its two books' grids; the
moves are non-overlapping `T_EXCL`-second windows, `(p_end / p_start - 1) * 1e4` bps. Both
directions of every selected pair are computed — the reverse direction is the reciprocal grid,
so its adverse (left) tail is the forward direction's right tail.

Moves are kept per day and split into the **fit sample (June)** and the **test sample (July)**.
1s klines were validated against raw tick data for exactly this use in the clustering notebook
(median cap difference < 0.3 bps). The first run downloads a few GB into
`data/binance_klines_1s/` and caches; later runs are offline. Books with no archive for a day
simply skip that day; pairs with no data in either period are dropped and reported.

In [ ]:
def closes_1s(symbol, day):
    """Close price per second of one UTC day, forward-filled onto the 86400-second grid."""
    path = os.path.join(KLINE_CACHE, f"{symbol}-1s-{day}.zip")
    if os.path.exists(path + ".missing"):
        return None
    if not os.path.exists(path):
        url = f"https://data.binance.vision/data/spot/daily/klines/{symbol}/1s/{symbol}-1s-{day}.zip"
        try:
            urllib.request.urlretrieve(url, path)
        except urllib.error.HTTPError:
            open(path + ".missing", "w").close()
            return None
    with zipfile.ZipFile(path) as z:
        df = pd.read_csv(z.open(z.namelist()[0]), header=None, usecols=[0, 4],
                         names=["ts", "close"])
    if not str(df.ts.iloc[0]).isdigit():                    # newer files ship a header row
        df = df.iloc[1:].astype({"ts": np.int64, "close": float})
    unit = 1_000_000 if df.ts.iloc[0] > 10**14 else 1_000   # timestamps: us (2025+) or ms
    sec_of_day = (df.ts.to_numpy(np.int64) // unit) % 86400
    px = np.full(86400, np.nan)
    px[sec_of_day] = df.close.to_numpy(float)
    return pd.Series(px).ffill().bfill().to_numpy()


def t_moves(grid):
    p = grid[::T_EXCL]
    return (p[1:] / p[:-1] - 1.0) * 1e4     # bps over one exclusivity window


def feeds_of(ukey):
    a, b = ukey.split("|")
    return f"{a}/{b}", f"{b}/{a}"           # forward (alphabetical) and reverse direction


FEEDS = sorted({f for u in usable.ukey for f in feeds_of(u)})
ASSETS = sorted({a for u in usable.ukey for a in u.split("|")})
DAY_MOVES = {f: {} for f in FEEDS}
for day in FIT_DAYS + TEST_DAYS:
    grids = {a: np.ones(86400) if a == "USDT" else closes_1s(a + "USDT", day)
             for a in ASSETS}
    for f in DAY_MOVES:
        s, b = f.split("/")
        if grids[s] is not None and grids[b] is not None:
            DAY_MOVES[f][day] = t_moves(grids[s] / grids[b])


def concat_moves(feed, days):
    have = [DAY_MOVES[feed][d] for d in days if d in DAY_MOVES[feed]]
    return np.concatenate(have) if have else None


MOVES_FIT = {f: concat_moves(f, FIT_DAYS) for f in FEEDS}
MOVES_TEST = {f: concat_moves(f, TEST_DAYS) for f in FEEDS}
ok = usable.ukey.map(lambda u: all(MOVES_FIT[f] is not None and MOVES_TEST[f] is not None
                                   for f in feeds_of(u)))
if (~ok).any():
    print(f"no price data in fit or test period, dropped: {sorted(usable.pair[~ok])}")
    usable = usable[ok].reset_index(drop=True)
print(f"{len(ASSETS)} Binance books, {len(usable)} pairs; "
      f"fit windows per direction: {min(len(MOVES_FIT[feeds_of(u)[0]]) for u in usable.ukey):,}+, "
      f"test: {min(len(MOVES_TEST[feeds_of(u)[0]]) for u in usable.ukey):,}+")

## 4. Fit: revert rate vs cap on June, and the two caps

Per pair, the revert rate at cap `c` is the attempt-weighted mix of the two directions' shares
of moves below `-c`. The group curve is the attempt-weighted average of pair curves over the
**June** sample, and the group's fixed cap is the smallest grid point where the curve is at or
below the target.

(For a single directed pair this is exactly the target-quantile of its moves with flipped
sign — the direct percentile method of the other cap notebooks. The weighted curve generalizes
it to a group's traffic mix.)

In [ ]:
CAP_GRID = np.arange(0.0, 50.0001, 0.05)    # candidate caps in bps


def revert_curve(moves):
    """Share of moves below -c, for every c in CAP_GRID."""
    return np.searchsorted(np.sort(moves), -CAP_GRID, side="left") / len(moves)


def pair_curve(row, moves):
    """Undirected pair curve: attempt-weighted mix of the two directions' tails."""
    fwd, rev = feeds_of(row.ukey)
    return (row.fwd * revert_curve(moves[fwd])
            + row.rev * revert_curve(moves[rev])) / row.attempts


usable["curve_fit"] = [pair_curve(r, MOVES_FIT) for r in usable.itertuples()]
usable["curve_test"] = [pair_curve(r, MOVES_TEST) for r in usable.itertuples()]

caps, cap_idx = {}, {}
fig, ax = plt.subplots(figsize=(8, 4.5))
for group, rows in usable.groupby("group"):
    weights = rows.attempts / rows.attempts.sum()
    curve = np.sum([w * c for w, c in zip(weights, rows.curve_fit)], axis=0)
    assert curve[-1] <= TARGET_REVERT_RATE, f"{group}: raise the CAP_GRID upper end"
    cap_idx[group] = int(np.argmax(curve <= TARGET_REVERT_RATE))
    caps[group] = CAP_GRID[cap_idx[group]]
    ax.plot(CAP_GRID, curve, label=f"{group}: cap = {caps[group]:.2f} bps")
    ax.axvline(caps[group], color=ax.lines[-1].get_color(), ls=":", lw=1)
ax.axhline(TARGET_REVERT_RATE, color="gray", ls="--", lw=1,
           label=f"target = {TARGET_REVERT_RATE:.0%}")
ax.set(xlabel="fixed penalty cap (bps of order size)",
       ylabel="expected price-driven revert rate",
       title=f"fit on June 2026: all chains, T = {T_EXCL}s, attempt-weighted",
       xlim=(0, 20), ylim=(0, 5 * TARGET_REVERT_RATE))
ax.legend()
plt.show()

In [ ]:
print(f"fit June 2026, all chains, T = {T_EXCL}s, "
      f"target price-driven revert rate = {TARGET_REVERT_RATE:.1%}\n")
usable["weight"] = usable.attempts / usable.groupby("group").attempts.transform("sum")
usable["rate_fit"] = [r.curve_fit[cap_idx[r.group]] for r in usable.itertuples()]
usable["rate_july"] = [r.curve_test[cap_idx[r.group]] for r in usable.itertuples()]
for group, rows in usable.groupby("group"):
    july = (rows.weight * rows.rate_july).sum()
    print(f"  {group:12s}: cap = {caps[group]:5.2f} bps  ->  "
          f"July out-of-sample rate {july:.2%}")

usable.sort_values("attempts", ascending=False).head(20)[
    ["pair", "group", "attempts", "weight", "rate_fit", "rate_july"]].round(4)

## 5. Out-of-sample check on July: over time, and per pair

Both figures use **July prices only**, at the caps fitted on June.

**Over time** — the daily expected revert rate at the fixed caps, weights held constant.
Movement here is purely market volatility: a fixed cap fitted on June delivers whatever July's
volatility implies. Sustained deviation from the target is the argument for an adaptive cap
("constant fitted k × current volatility estimate", as in the empirical-approach notebooks).

**Per pair** — the July revert rate each selected pair experiences at its group's cap.
Dispersion here is the cost of one cap per group: quiet majors sit below the target and
volatile long-tail pairs far above it. A wide spread within "uncorrelated" is the case for a
third tier (or per-pair caps).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for group, rows in usable.groupby("group"):
    num, den = {}, {}
    for r in rows.itertuples():
        fwd, rev = feeds_of(r.ukey)
        thr = -caps[group]
        for day in TEST_DAYS:
            if day in DAY_MOVES[fwd]:
                hit = r.fwd * (DAY_MOVES[fwd][day] < thr).mean() \
                    + r.rev * (DAY_MOVES[rev][day] < thr).mean()
                num[day] = num.get(day, 0.0) + hit
                den[day] = den.get(day, 0.0) + r.attempts
    daily = pd.Series({pd.Timestamp(d): num[d] / den[d] for d in num}).sort_index()
    ax.plot(daily.index, daily.values, label=f"{group} (cap = {caps[group]:.2f} bps)")
ax.axhline(TARGET_REVERT_RATE, color="gray", ls="--", lw=1,
           label=f"target = {TARGET_REVERT_RATE:.0%}")
ax.set(ylabel="expected price-driven revert rate",
       title="July 2026 (out-of-sample): daily revert rate at the June-fitted caps")
ax.legend()
fig.autofmt_xdate()
plt.show()

In [ ]:
top = usable.sort_values("rate_july")
colors = {"correlated": "C0", "uncorrelated": "C1"}

fig, ax = plt.subplots(figsize=(8, max(5.0, 0.22 * len(top))))
ax.barh(range(len(top)), top.rate_july, color=top.group.map(colors))
ax.set_yticks(range(len(top)), top.pair, fontsize=7)
ax.axvline(TARGET_REVERT_RATE, color="gray", ls="--", lw=1)
ax.set(xlabel="July revert rate at the group's June-fitted cap",
       title=f"July 2026 (out-of-sample): all {len(top)} selected pairs")
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c, label=g) for g, c in colors.items()]
          + [plt.Line2D([], [], color="gray", ls="--",
                        label=f"target = {TARGET_REVERT_RATE:.0%}")])
plt.tight_layout()
plt.show()

## 6. Comparison with the current caps (fixed native amount per chain)

Today the cap is a fixed native-token amount per chain (`penalty_cap_native`, constant per
extract). In bps of order size it is therefore *size-dependent*: `cap_native / volume_native`
(unit-free, so comparable across chains). Small orders get an enormous bps cap, whales a tiny
one.

To compare regimes on equal footing we ask, for every attempt on a selected pair: *what
price-driven revert probability does its cap imply on July prices?* — reading the attempt's
pair test-curve at the attempt's cap (current: its individual bps equivalent; proposed: its
group's fixed cap). Averaging gives the expected revert rate under each regime — overall, per
chain, and by USD order size.

In [ ]:
att2 = att.merge(mapped[["chain", "sell_token", "buy_token", "ukey", "group"]],
                 on=["chain", "sell_token", "buy_token"])
att2 = att2.merge(usable[["ukey", "group", "rate_july"]], on=["ukey", "group"])
att2 = att2[att2.volume_native > 0].copy()
att2["cap_now_bps"] = att2.penalty_cap_native / att2.volume_native * 1e4
curve_of = {(r.ukey, r.group): r.curve_test for r in usable.itertuples()}
att2["p_now"] = np.nan
for (u, g), rows in att2.groupby(["ukey", "group"]):
    att2.loc[rows.index, "p_now"] = np.interp(rows.cap_now_bps, CAP_GRID, curve_of[(u, g)])

print(f"expected price-driven revert rate on July prices, selected pairs "
      f"({len(att2):,} attempts):")
print(f"  current fixed native caps  : {att2.p_now.mean():.2%}")
print(f"  proposed two fixed bps caps: {att2.rate_july.mean():.2%}"
      f"   (target {TARGET_REVERT_RATE:.0%})\n")
per_chain = att2.groupby("chain").agg(
    attempts=("p_now", "size"),
    cap_native=("penalty_cap_native", lambda s: s.iloc[0] / 1e18),
    cap_bps_median=("cap_now_bps", "median"),
    current=("p_now", "mean"), proposed=("rate_july", "mean"))
per_chain["native"] = [NATIVE[c] for c in per_chain.index]
print(per_chain.round(4).to_string())

sized = att2[att2.usd.notna()].copy()
sized["decile"] = pd.qcut(sized.usd, 10, labels=False, duplicates="drop")
by_size = sized.groupby("decile").agg(usd=("usd", "median"),
                                      current=("p_now", "mean"),
                                      proposed=("rate_july", "mean"))
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(by_size.usd, by_size.current, "o--", color="C0",
        label="current caps (fixed native amount per chain)")
ax.plot(by_size.usd, by_size.proposed, "o-", color="C1",
        label="proposed caps (fixed bps of size)")
ax.axhline(TARGET_REVERT_RATE, color="gray", ls="--", lw=1,
           label=f"target = {TARGET_REVERT_RATE:.0%}")
ax.set(xscale="log", xlabel="order size (decile median, USD)",
       ylabel="expected price-driven revert rate",
       title="revert rate by order size on July prices, current vs proposed cap")
ax.legend()
plt.show()

## 7. Caveats — read before quoting the numbers

1. **Price-driven reverts only.** Realized revert rates carry a large non-price component
   (ethereum Jan–Jun 2026: ~11.6% on this same frame), far above any price-driven target. If
   the target is meant for *total* reverts, subtract the external share `p_ext` first: price
   allowance = `(q - p_ext) / (1 - p_ext)` (8pct notebook, section 7).
2. **Same-book pairs are excluded from the curves** (`WETH↔wstETH`, `DAI↔USDC`, `WBTC↔cbBTC`,
   ...): their legs resolve to one Binance book, so the model has no independent price for the
   cross. The listed pegged books that do exist (`USDCUSDT`, `WBETHETH`) are tick-bound at
   ~0.1 bps, so their price-driven revert risk is a tick floor — while realized WETH↔wstETH
   revert rates are ~50%, i.e. overwhelmingly non-price-driven. A small correlated cap is what
   the price model justifies; it will not deter whatever actually drives those reverts.
3. **CMS buckets are taken as-is.** The stables bucket mixes USD stables, EUR stables and
   tokenized equities, so e.g. a EUR↔USD stable pair counts as correlated while carrying real
   FX volatility; there is no BTC bucket, so BTC wrappers are correlated in spirit but land in
   "uncorrelated" (and are excluded as same-book pairs anyway).
4. **Coverage is bounded by the long tail.** Section 2 quantifies it: ~90% of USD volume sits
   in a few dozen pooled pairs, but 90% of *trades* needs on the order of a thousand — no
   hand-maintained list gets there, and the top-`TOP_PAIRS` selection covers the printed share
   of attempts only. The unmapped tail is likely *more* volatile, and the per-pair figure
   shows how wide dispersion already is inside the selection (the case for a third tier, and
   the clustering notebook's territory).
5. **Direction mixing.** Both directions of a pair share one curve (their attempt-weighted
   mix), so the per-attempt current-cap comparison ignores which direction an attempt traded.
   For near-symmetric short-horizon moves the difference is small; stable pairs show it most
   (the two USDC↔USDT tails differ by the off-peg side).
6. **Attempt weights predate the test period**: the extracts end in June, so July revert rates
   are evaluated with the Jan–Jun pair mix; July's actual flow mix may differ.
7. **Synthetic crosses overstate volatility** (independent leg noise adds instead of
   cancelling — `synth_vs_direct_vol` notebook), so volatile/volatile pairs err toward a
   conservative cap.
8. **No bidding response.** Solvers shade bids when caps change (tie-equilibrium model in the
   `revert-rate-vs-cap` branch and 8pct section 7); over 1–10 bps caps the bid premium stays
   below ~0.5 bps, so the effect is second-order and omitted.
9. **USD sizes are approximate**: `volume_native x` Binance daily native close (xDAI ~ 1);
   corrupt `volume_native` rows (> $1B) are excluded from USD-based views. Revert
   probabilities in section 6 are read off curves that end at 50 bps (attempts whose current
   cap exceeds that — small orders — are assigned the 50 bps rate).

Relation to the other notebooks: this one produces *fixed* per-group caps with a June-fit /
July-test split; `penalty_cap_empirical_approach{,_8pct}` produce per-pair adaptive caps
(`k·σ·√T` and rolling percentiles); `solver_revert_option_pricing` derives the theory linking
cap and revert probability; `clustering_analysis_T26` assigns caps across the whole token
universe; `synth_vs_direct_vol` quantifies the synthetic-feed error used in caveat 7.